## 5-6. Regularized Linear Models – Ridge, Lasso
### Regularized Linear Model - Ridge Regression
### 보스턴 집값을 이어서 하는 회귀 연습이라서 코드만 적고 설명만 적어서 마무리

In [10]:

# [Ridge 회귀 기본 실행]
# Ridge: 일반 선형 회귀에 L2 정규화(패널티)를 추가한 모델
# alpha가 클수록 계수(coeff)를 더 강하게 축소 → 과적합 방지


# 앞의 LinearRegression예제에서 분할한 feature 데이터 셋인 X_data과 Target 데이터 셋인 Y_target 데이터셋을 그대로 이용 
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score



ridge = Ridge(alpha = 10)  # alpha=10: 정규화 강도 설정 (클수록 계수를 더 많이 줄임)

# cross_val_score: 5-fold 교차검증으로 모델 성능 평가
# scoring='neg_mean_squared_error': MSE를 음수로 반환 (sklearn 내부 규칙)
neg_mse_scores = cross_val_score(ridge, X_data, y_target, scoring="neg_mean_squared_error", cv = 5)

# MSE가 음수로 나오므로 -1을 곱해 양수로 변환 후 제곱근 → RMSE
rmse_scores  = np.sqrt(-1 * neg_mse_scores)
avg_rmse = np.mean(rmse_scores)  # 5개 fold의 RMSE 평균

print(' 5 folds 의 개별 Negative MSE scores: ', np.round(neg_mse_scores, 3))
print(' 5 folds 의 개별 RMSE scores : ', np.round(rmse_scores,3))
print(' 5 folds 의 평균 RMSE : {0:.3f} '.format(avg_rmse))


NameError: name 'X_data' is not defined

In [ ]:

# [alpha 값 변화에 따른 Ridge 성능 비교]
# alpha가 0이면 일반 선형 회귀와 동일, 커질수록 정규화 강해짐


# 릿지에 사용될 alpha 파라미터의 값을 정의
alphas = [0, 0.1, 1, 10, 100]

# alphas list 값을 반복하면서 alpha에 따른 평균 rmse를 구함.
for alpha in alphas :
    ridge = Ridge(alpha = alpha)
    
    # cross_val_score를 이용해 5 폴드의 평균 RMSE를 계산
    neg_mse_scores = cross_val_score(ridge, X_data, y_target, scoring="neg_mean_squared_error", cv = 5)
    avg_rmse = np.mean(np.sqrt(-1 * neg_mse_scores))
    print('alpha {0} 일 때 5 folds 의 평균 RMSE : {1:.3f} '.format(alpha, avg_rmse))

NameError: name 'X_data' is not defined

In [ ]:
# [alpha에 따른 회귀 계수 시각화]
# alpha가 커질수록 각 feature의 계수(coefficient)가 0에 가까워지는 것을 막대그래프로 확인
# Ridge는 계수를 완전히 0으로 만들지 않고 축소만 함 (Lasso와의 차이)

# 각 alpha에 따른 회귀 계수 값을 시각화하기 위해 5개의 열로 된 맷플롯립 축 생성  
fig , axs = plt.subplots(figsize=(18,6) , nrows=1 , ncols=5)

# 각 alpha에 따른 회귀 계수 값을 데이터로 저장하기 위한 DataFrame 생성  
coeff_df = pd.DataFrame()

# enumerate: 반복할 때 인덱스(pos)와 값(alpha)을 동시에 얻음
# alphas 리스트 값을 차례로 입력해 회귀 계수 값 시각화 및 데이터 저장. pos는 axis의 위치 지정
for pos , alpha in enumerate(alphas) :
    ridge = Ridge(alpha = alpha)
    ridge.fit(X_data , y_target)  # 전체 데이터로 학습

    # ridge.coef_: 학습된 각 feature의 회귀 계수 배열
    # alpha에 따른 피처별 회귀 계수를 Series로 변환하고 이를 DataFrame의 컬럼으로 추가.  
    coeff = pd.Series(data=ridge.coef_ , index=X_data.columns )
    colname='alpha:'+str(alpha)
    coeff_df[colname] = coeff

    # 계수를 내림차순 정렬 후 가로 막대 그래프 출력
    # 막대 그래프로 각 alpha 값에서의 회귀 계수를 시각화. 회귀 계수값이 높은 순으로 표현
    coeff = coeff.sort_values(ascending=False)
    axs[pos].set_title(colname)
    axs[pos].set_xlim(-3,6)  # x축 범위 고정 → 그래프 간 비교 용이
    sns.barplot(x=coeff.values , y=coeff.index, ax=axs[pos])

# for 문 바깥에서 맷플롯립의 show 호출 및 alpha에 따른 피처별 회귀 계수를 DataFrame으로 표시
plt.show()



In [ ]:

# [alpha별 회귀 계수를 표 형태로 출력]
# coeff_df: 각 열이 alpha 값, 각 행이 feature의 회귀 계수
# alpha=0 기준으로 내림차순 정렬하면 중요 feature 파악 가능


ridge_alphas = [0 , 0.1 , 1 , 10 , 100]
sort_column = 'alpha:'+str(ridge_alphas[0])  # 'alpha:0' 컬럼 기준으로 정렬
coeff_df.sort_values(by=sort_column, ascending=False)  # 회귀 계수 큰 순서로 정렬

NameError: name 'coeff_df' is not defined

In [ ]:

# [Ridge / Lasso / ElasticNet 통합 평가 함수 정의]
# Lasso: L1 정규화 → 일부 계수를 정확히 0으로 만들어 feature selection 효과
# ElasticNet: L1 + L2 혼합 (l1_ratio로 비율 조절)


from sklearn.linear_model import Lasso, ElasticNet

# alpha값에 따른 회귀 모델의 폴드 평균 RMSE를 출력하고 회귀 계수값들을 DataFrame으로 반환 
def get_linear_reg_eval(model_name, params=None, X_data_n=None, y_target_n=None, 
                        verbose=True, return_coeff=True):
    coeff_df = pd.DataFrame()
    if verbose : print('####### ', model_name , '#######')
    for param in params:
        # model_name 문자열에 따라 해당 모델 객체 생성
        if model_name =='Ridge': model = Ridge(alpha=param)
        elif model_name =='Lasso': model = Lasso(alpha=param)
        elif model_name =='ElasticNet': model = ElasticNet(alpha=param, l1_ratio=0.7)  # L1 비율 70% 고정

        neg_mse_scores = cross_val_score(model, X_data_n, 
                                             y_target_n, scoring="neg_mean_squared_error", cv = 5)
        avg_rmse = np.mean(np.sqrt(-1 * neg_mse_scores))
        print('alpha {0}일 때 5 폴드 세트의 평균 RMSE: {1:.3f} '.format(param, avg_rmse))

        # cross_val_score는 evaluation metric만 반환하므로 모델을 다시 학습하여 회귀 계수 추출
        # → cross_val_score 내부에서 학습된 모델은 외부에서 접근 불가 → 별도로 fit() 필요
        model.fit(X_data_n , y_target_n)
        if return_coeff:
            # alpha에 따른 피처별 회귀 계수를 Series로 변환하고 이를 DataFrame의 컬럼으로 추가. 
            coeff = pd.Series(data=model.coef_ , index=X_data_n.columns )
            colname='alpha:'+str(param)
            coeff_df[colname] = coeff
    
    return coeff_df
# end of get_linear_regre_eval

In [ ]:

# [Lasso 모델 성능 평가]
# Lasso는 alpha가 크면 일부 feature 계수가 완전히 0이 됨
# → 불필요한 변수를 자동으로 제거하는 효과 (sparse model)

# 라쏘에 사용될 alpha 파라미터의 값들을 정의하고 get_linear_reg_eval() 함수 호출
lasso_alphas = [ 0.07, 0.1, 0.5, 1, 3]
coeff_lasso_df =get_linear_reg_eval('Lasso', params=lasso_alphas, X_data_n=X_data, y_target_n=y_target)

NameError: name 'get_linear_reg_eval' is not defined

In [ ]:

# [Lasso 회귀 계수 표 출력]
# alpha가 커질수록 0이 되는 계수가 늘어나는 것을 확인


# 반환된 coeff_lasso_df를 첫번째 컬럼순으로 내림차순 정렬하여 회귀계수 DataFrame출력
sort_column = 'alpha:'+str(lasso_alphas[0])  # 가장 작은 alpha 기준으로 정렬
coeff_lasso_df.sort_values(by=sort_column, ascending=False)

In [ ]:

# [ElasticNet 모델 성능 평가]
# ElasticNet = Ridge(L2) + Lasso(L1)의 혼합
# l1_ratio=0.7 → L1 패널티 70%, L2 패널티 30% 적용
# 상관관계 높은 feature가 많을 때 Lasso보다 안정적


# 엘라스틱넷에 사용될 alpha 파라미터의 값들을 정의하고 get_linear_reg_eval() 함수 호출
# l1_ratio는 0.7로 고정
elastic_alphas = [ 0.07, 0.1, 0.5, 1, 3]
coeff_elastic_df =get_linear_reg_eval('ElasticNet', params=elastic_alphas,
                                      X_data_n=X_data, y_target_n=y_target)

In [ ]:

# [ElasticNet 회귀 계수 표 출력]


# 반환된 coeff_elastic_df를 첫번째 컬럼순으로 내림차순 정렬하여 회귀계수 DataFrame출력
sort_column = 'alpha:'+str(elastic_alphas[0])
coeff_elastic_df.sort_values(by=sort_column, ascending=False)

NameError: name 'elastic_alphas' is not defined

In [ ]:

# [데이터 스케일링 함수 정의]
# 정규화 방법에 따라 다른 전처리 적용 후 선택적으로 다항식 특성 추가
# - Standard: 평균 0, 표준편차 1로 변환 (z-score 정규화)
# - MinMax: 0~1 범위로 변환
# - Log: log(1+x) 변환 → 오른쪽으로 치우친 분포에 효과적
# - PolynomialFeatures: x1, x2 → x1, x2, x1², x1*x2, x2² 등 추가


from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures

# method는 표준 정규 분포 변환(Standard), 최대값/최소값 정규화(MinMax), 로그변환(Log) 결정
# p_degree는 다향식 특성을 추가할 때 적용. p_degree는 2이상 부여하지 않음. 
def get_scaled_data(method='None', p_degree=None, input_data=None):
    if method == 'Standard':
        scaled_data = StandardScaler().fit_transform(input_data)  # 평균 0, 분산 1
    elif method == 'MinMax':
        scaled_data = MinMaxScaler().fit_transform(input_data)    # 0~1 범위
    elif method == 'Log':
        scaled_data = np.log1p(input_data)  # log(1+x): 0값 처리 가능, 음수 방지
    else:
        scaled_data = input_data  # 변환 없이 원본 그대로 사용

    if p_degree != None:
        # 다항식 특성 추가 (include_bias=False: 절편 항 제외)
        scaled_data = PolynomialFeatures(degree=p_degree, 
                                         include_bias=False).fit_transform(scaled_data)
    
    return scaled_data

In [ ]:

# [6가지 스케일링 조합 × Ridge alpha 5개 = 30개 실험]
# 어떤 전처리 방법이 Ridge 성능을 가장 높이는지 비교
# (None, None): 원본 데이터 그대로
# ('Standard', 2): 표준화 후 2차 다항식 특성 추가


#변환 방법은 모두 6개, 원본 그대로, 표준정규분포, 표준정규분포+다항식 특성
# 최대/최소 정규화, 최대/최소 정규화+다항식 특성, 로그변환 
scale_methods=[(None, None), ('Standard', None), ('Standard', 2), 
               ('MinMax', None), ('MinMax', 2), ('Log', None)]

for scale_method in scale_methods:
    # 각 스케일링 조합으로 X_data 변환
    X_data_scaled = get_scaled_data(method=scale_method[0], p_degree=scale_method[1], 
                                    input_data=X_data)
    print(X_data_scaled.shape, X_data.shape)  # 다항식 추가 시 feature 수가 늘어남을 확인
    print('\n## 변환 유형:{0}, Polynomial Degree:{1}'.format(scale_method[0], scale_method[1]))

    # verbose=False: 모델명 헤더 출력 생략 / return_coeff=False: 계수 반환 생략
    get_linear_reg_eval('Ridge', params=alphas, X_data_n=X_data_scaled, 
                        y_target_n=y_target, verbose=False, return_coeff=False)

NameError: name 'X_data' is not defined

## 로지스틱 회귀

In [9]:

# [데이터 로드]
# breast cancer 데이터: 유방암 양성/음성 이진 분류 문제
# sklearn 내장 데이터셋, 569개 샘플 × 30개 feature


import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()  # target: 0=악성(malignant), 1=양성(benign)

In [ ]:

# [데이터 전처리 및 분할]
# 로지스틱 회귀는 feature 스케일에 민감 → StandardScaler 필수
# train 70% / test 30% 분할


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# StandardScaler( )로 평균이 0, 분산 1로 데이터 분포도 변환
scaler = StandardScaler()
data_scaled = scaler.fit_transform(cancer.data)  # 전체 데이터를 표준화

# random_state=0: 동일한 분할 재현 보장
X_train , X_test, y_train , y_test = train_test_split(data_scaled, cancer.target, test_size=0.3, random_state=0)

In [12]:

# [로지스틱 회귀 학습 및 평가]
# accuracy: 전체 정확도
# roc_auc: 양성/음성 구별 능력 (1에 가까울수록 좋음)
# predict_proba: 클래스별 확률 반환 → [:, 1]은 양성(1) 확률만 추출


from sklearn.metrics import accuracy_score, roc_auc_score

# 로지스틱 회귀를 이용하여 학습 및 예측 수행. 
# solver인자값을 생성자로 입력하지 않으면 solver='lbfgs'  
lr_clf = LogisticRegression() # solver='lbfgs'
lr_clf.fit(X_train, y_train)  # 학습
lr_preds = lr_clf.predict(X_test)               # 클래스 예측 (0 또는 1)
lr_preds_proba = lr_clf.predict_proba(X_test)[:, 1]  # 양성 확률값 (roc_auc 계산용)

# accuracy와 roc_auc 측정
print('accuracy: {0:.3f}, roc_auc:{1:.3f}'.format(accuracy_score(y_test, lr_preds),
                                                 roc_auc_score(y_test , lr_preds_proba)))

accuracy: 0.977, roc_auc:0.995


In [13]:

# [solver 종류별 성능 비교]
# solver: 로지스틱 회귀의 최적화 알고리즘 (비용함수 최소화 방법)
# - lbfgs: 기본값, 소규모 데이터에 적합
# - liblinear: 소규모 데이터, L1/L2 모두 지원
# - newton-cg / sag / saga: 대규모 데이터에 유리
# max_iter: 수렴할 때까지 반복 횟수 상한


solvers = ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']
# 여러개의 solver값 별로 LogisticRegression 학습 후 성능 평가
for solver in solvers:
    lr_clf = LogisticRegression(solver=solver, max_iter=600)  # 반복 횟수 600으로 설정 (수렴 보장)
    lr_clf.fit(X_train, y_train)
    lr_preds = lr_clf.predict(X_test)
    lr_preds_proba = lr_clf.predict_proba(X_test)[:, 1]

    # accuracy와 roc_auc 측정
    print('solver:{0}, accuracy: {1:.3f}, roc_auc:{2:.3f}'.format(solver, 
                                                                  accuracy_score(y_test, lr_preds),
                                                                  roc_auc_score(y_test , lr_preds_proba)))                              

solver:lbfgs, accuracy: 0.977, roc_auc:0.995
solver:liblinear, accuracy: 0.982, roc_auc:0.995
solver:newton-cg, accuracy: 0.977, roc_auc:0.995
solver:sag, accuracy: 0.982, roc_auc:0.995
solver:saga, accuracy: 0.982, roc_auc:0.995


In [ ]:

# [GridSearchCV로 최적 하이퍼파라미터 탐색]
# solver × penalty × C 조합 2×2×5=20가지를 cv=3으로 교차검증
# C: 정규화 강도의 역수 (C 작을수록 강한 정규화 = alpha 클수록과 같은 효과)
# penalty: L1(Lasso식) 또는 L2(Ridge식) 정규화
# 주의: lbfgs는 L1을 지원 안 해서 FitFailedWarning 발생 (정상)


from sklearn.model_selection import GridSearchCV

params={'solver':['liblinear', 'lbfgs'],
        'penalty':['l2', 'l1'],
        'C':[0.01, 0.1, 1, 5, 10]}  # C: 작을수록 강한 정규화

lr_clf = LogisticRegression()

# GridSearchCV: 모든 파라미터 조합을 완전 탐색 (brute-force)
grid_clf = GridSearchCV(lr_clf, param_grid=params, scoring='accuracy', cv=3 )
grid_clf.fit(data_scaled, cancer.target)  # 전체 데이터로 탐색

# best_params_: 최적 조합 / best_score_: 해당 조합의 CV 평균 정확도
print('최적 하이퍼 파라미터:{0}, 최적 평균 정확도:{1:.3f}'.format(grid_clf.best_params_, 
                                                  grid_clf.best_score_))

최적 하이퍼 파라미터:{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}, 최적 평균 정확도:0.979


c:\Users\hyoli\anaconda\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
15 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\hyoli\anaconda\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hyoli\anaconda\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\hyoli\anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1218, in fit
    solver = _chec